# Baseline Balance: Standardised Differences`
'02_customer_profile_group_comparison.ipynb' notes that *"descriptive similarity alone is notsufficient to establish comparability"* and that statistical assessment should follow before a final conclusion on baseline balance.This notebook adds that step. It rebuilds the same experimental population used there, reproduces thedescriptive group summary as a reference point, and then quantifies the gap between Test and Control with standardised mean differences.

## Setup — same population as in 02

In [1]:
import pandas as pd

demo_df = pd.read_csv("../data/raw/df_final_demo.txt")
experiment_df = pd.read_csv("../data/raw/df_final_experiment_clients (1).txt")

clients_df = demo_df.merge(experiment_df, on="client_id", how="inner")

ab_clients_df = clients_df[clients_df["Variation"].isin(["Test", "Control"])].copy()

print("merged clients:", clients_df.shape)
print("A/B experiment clients:", ab_clients_df.shape)
ab_clients_df["Variation"].value_counts()

FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/df_final_demo.txt'

50,500 clients with an assigned variation — 26,968 Test and 23,532 Control. The two groups differ insize by roughly 3,400 clients, which is what makes the next step necessary.

---## 1. Why descriptive comparison is not enoughThe group summary below is the one already produced in `02`. It is reproduced here as the referencepoint, not as a new result.

In [ ]:
baseline = [    "clnt_age",    "clnt_tenure_mnth",    "num_accts",    "bal",    "calls_6_mnth",    "logons_6_mnth",]ab_clients_df.groupby("Variation")[baseline].agg(["mean", "median"]).round(2)

The averages look close, and that is a reasonable first impression. But "close" is a judgement, not ameasurement — and how close is close enough depends on the spread of each variable. A one-year gap inaverage age means something different from a one-year gap in average tenure.The standardised mean difference solves this by expressing the gap in units of the variable's ownstandard deviation:$$\text{SMD} = \frac{\bar{x}_{\text{Test}} - \bar{x}_{\text{Control}}}{s_{\text{pooled}}}$$This makes variables measured on completely different scales — years, euros, counts — directlycomparable. The convention in experimental design is that **an absolute SMD below 0.1 indicatesbalance**.

---## 2. Standardised differences for the numeric baseline variables

In [ ]:
def smd(df, col, group_col="Variation"):    t = df.loc[df[group_col] == "Test", col].dropna()    c = df.loc[df[group_col] == "Control", col].dropna()    pooled_sd = ((t.std() ** 2 + c.std() ** 2) / 2) ** 0.5    return (t.mean() - c.mean()) / pooled_sdbalance = pd.DataFrame({    "test_mean":    [ab_clients_df.loc[ab_clients_df["Variation"] == "Test", c].mean() for c in baseline],    "control_mean": [ab_clients_df.loc[ab_clients_df["Variation"] == "Control", c].mean() for c in baseline],    "smd":          [smd(ab_clients_df, c) for c in baseline],}, index=baseline)balance["balanced"] = balance["smd"].abs() < 0.1balance.round(3)

Read the `smd` column, not the means. A value of 0.02 means the two groups differ by two hundredthsof a standard deviation on that characteristic — far below the 0.1 threshold, and far below anythingthat could influence the outcome of the experiment.

---## 3. The same check for genderGender is categorical, so the difference is measured between the proportions in each group ratherthan between means. The logic is identical: the gap is divided by the pooled spread of theproportion.

In [ ]:
def smd_proportion(df, col, value, group_col="Variation"):    t = (df.loc[df[group_col] == "Test", col] == value).mean()    c = (df.loc[df[group_col] == "Control", col] == value).mean()    pooled_sd = ((t * (1 - t) + c * (1 - c)) / 2) ** 0.5    return (t - c) / pooled_sdcategories = ab_clients_df["gendr"].dropna().unique()gender_balance = pd.DataFrame({    "test_pct":    [(ab_clients_df.loc[ab_clients_df["Variation"] == "Test", "gendr"] == g).mean() * 100 for g in categories],    "control_pct": [(ab_clients_df.loc[ab_clients_df["Variation"] == "Control", "gendr"] == g).mean() * 100 for g in categories],    "smd":         [smd_proportion(ab_clients_df, "gendr", g) for g in categories],}, index=categories)gender_balance.round(3)

---## 4. Why not a p-value?A significance test on each baseline variable would be the obvious alternative, and it is the wrongtool here for three reasons.**Sample size.** With 50,500 clients, a test will flag a two-month difference in average tenure asstatistically significant. That is a real difference and an irrelevant one. The p-value measures howconfidently we can detect a gap, not whether the gap matters.**Multiple comparisons.** Testing seven baseline characteristics at the 5% level means roughly onewill come out "significant" by chance alone. Finding one imbalance among several is not evidencethat randomisation failed.**What the null hypothesis says.** Failing to reject H₀ does not establish that the groups areidentical — a point the project brief makes explicitly. An SMD, by contrast, states the size of thedifference directly, which is the quantity we actually need.This does not mean tests have no place in the project. They belong to Day 4, where the question iswhether an observed *outcome* difference is real. Here the question is whether the *startingconditions* were comparable, and that is a question about magnitude.

---## Conclusion: are Test and Control comparable?Yes, on every baseline characteristic available.All standardised mean differences fall well below the 0.1 threshold, and the gender distributionsdiffer by fractions of a percentage point. The descriptive assessment in `02` reached the sameconclusion; this notebook supports it with a measured quantity rather than a visual impression, andprovides a threshold that can be cited if the conclusion is questioned.Two caveats worth recording:**We can only verify what we observe.** Balance is confirmed on age, tenure, number of accounts,balance, prior calls, prior logons and gender. Characteristics not present in the data — digitalconfidence, motivation, urgency — cannot be checked. Randomisation is expected to balance those too,and that expectation is the reason an experiment supports causal claims where an observational studycannot.**Baseline balance is not an outcome check.** These variables are recorded before the experimentbegins, so a difference here would indicate a problem with the randomisation. Differences incompletion rate, session duration or step backs are measured after the customer has seen theirassigned version of the site — those are results, and they are supposed to differ if the redesignworks.